# Co-factor swap

Author: Vitor Pereira

License: [CC BY-SA 4.0](http://creativecommons.org/licenses/by-sa/4.0/)

-------

This example searches for co-factor swaps that increase the theoretical L-threonine production of *E. coli* (model iJO1366). A swap changes the co-factor specificity of an oxidoreductase from NAD(H) to NADP(H), the strategy of OptSwap (King and Feist, 2013).

Reference reading:
[https://doi.org/10.1016/j.ymben.2014.05.009](https://doi.org/10.1016/j.ymben.2014.05.009).

The optimization uses small settings (population size and number of generations) so that the notebook runs in a few minutes.

In [1]:
import logging
import warnings

import pandas as pd
from cobra.io import read_sbml_model

import mewpy
from mewpy.optimization import EA
from mewpy.optimization.evaluation import TargetFlux
from mewpy.problems import CofactorSwapProblem

# MEWpy logs the fitness of each generation at INFO level; jMetalPy logs at DEBUG level by default.
mewpy.configure_logging("INFO")
logging.getLogger("jmetal").setLevel(logging.WARNING)
# Candidates that make the model infeasible are expected during the search (they get the worst fitness).
warnings.filterwarnings("ignore", message="Solver status is 'infeasible'")

## Model and conditions

The model is loaded with COBRApy. The cell below closes formate-hydrogen lyase (`FHL`), catalase (`CAT`) and the superoxide dismutases (`SPODM`, `SPODMpp`), which are already closed in this SBML file, makes pyruvate synthase (`POR5`) irreversible, and limits the glucose and oxygen uptakes to 10 mmol/gDW/h.

In [2]:
model = read_sbml_model('models/ec/iJO1366.xml')

for rid in ['FHL', 'CAT', 'SPODM', 'SPODMpp']:
    model.reactions.get_by_id(rid).knock_out()
model.reactions.POR5.lower_bound = 0
model.reactions.EX_glc__D_e.lower_bound = -10
model.reactions.EX_o2_e.lower_bound = -10

BIOMASS = 'BIOMASS_Ec_iJO1366_core_53p95M'
PRODUCT = 'EX_thr__L_e'

## The co-factor swap problem

`CofactorSwapProblem` works on a copy of the model. Its targets are the cytosolic reactions that use NAD+/NADH and not NADP+/NADPH. For each of them it adds a copy that uses NADP+/NADPH instead, closed by default. Swapping a target closes the original reaction and opens the copy with the original bounds.

The objective is `TargetFlux(PRODUCT)`, the threonine export flux in a pFBA simulation. The cells below make threonine export the simulation objective and require a growth rate of at least 0.1 1/h, so the fitness is the threonine production at that minimal growth, kept by pFBA at 99.9% of its maximum (a theoretical capacity, not growth-coupled production). These changes are made through `problem.simulator`, so they apply to the problem's copy of the model.

In [3]:
problem = CofactorSwapProblem(model, [TargetFlux(PRODUCT)], candidate_max_size=4)
problem.simulator.set_reaction_bounds(BIOMASS, 0.1, 1000)
problem.simulator.objective = PRODUCT

print('targets: %d reactions, e.g.' % len(problem.target_list), problem.target_list[:5])
baseline = problem.simulate(solution={}, method='pFBA')
print('without swaps: threonine %.4f mmol/gDW/h at growth %.2f 1/h'
      % (baseline.fluxes[PRODUCT], baseline.fluxes[BIOMASS]))

targets: 116 reactions, e.g. ['2DGULRGx', '2DGULRx', 'ABUTD', '3HCINNMH', '3HPPPNH']


without swaps: threonine 11.5342 mmol/gDW/h at growth 0.10 1/h


## Optimization

With a single objective, the EA is a genetic algorithm (GA). It runs 10 generations of 50 candidates (500 evaluations) in this process (`mp=False`), with a fixed `seed` for its random choices; another run can return a different solution. Candidates swap 1 to 4 reactions, and `run()` simplifies the final solutions by removing the swaps that do not change the fitness.

In [4]:
ea = EA(problem, max_generations=10, population_size=50, mp=False, seed=42)
final_population = ea.run()

[2026-11-09 23:20:48] [mewpy.optimization.jmetal.ea] [INFO] Running GA


[2026-11-09 23:20:56] [mewpy] [INFO] Evaluations: 50	Fitness: 11.53421499552716


[2026-11-09 23:21:05] [mewpy] [INFO] Evaluations: 100	Fitness: 12.070357503052197


[2026-11-09 23:21:15] [mewpy] [INFO] Evaluations: 150	Fitness: 12.070357503052202


[2026-11-09 23:21:27] [mewpy] [INFO] Evaluations: 200	Fitness: 12.070357503052465


[2026-11-09 23:21:38] [mewpy] [INFO] Evaluations: 250	Fitness: 12.070357503052465


[2026-11-09 23:21:48] [mewpy] [INFO] Evaluations: 300	Fitness: 12.070357503052465


[2026-11-09 23:21:59] [mewpy] [INFO] Evaluations: 350	Fitness: 12.070357503052552


[2026-11-09 23:22:10] [mewpy] [INFO] Evaluations: 400	Fitness: 12.070357503052552


[2026-11-09 23:22:20] [mewpy] [INFO] Evaluations: 450	Fitness: 12.070357503052552


[2026-11-09 23:22:30] [mewpy] [INFO] Evaluations: 500	Fitness: 12.070357503052552


In [5]:
ea.dataframe()

,Modification,Size,TargetFlux
0,{'TRSARr': 0},1,12.070358


## Inspecting the best solution

The best solution is simulated with its swaps. The table lists the reactions that produce NADPH, with and without the swaps (fluxes in mmol/gDW/h, times the stoichiometric coefficient of NADPH).

In [6]:
best = max(final_population, key=lambda s: s.fitness[0])
print('swapped:', list(best.values))
print('constraints:', best.constraints)
res = problem.simulate(solution=best.values, method='pFBA')
for name, result in [('without swaps', baseline), ('with swaps', res)]:
    print('%s: threonine %.4f mmol/gDW/h, growth %.2f 1/h, glucose uptake %.1f mmol/gDW/h'
          % (name, result.fluxes[PRODUCT], result.fluxes[BIOMASS], -result.fluxes['EX_glc__D_e']))


def producers(result, metabolite='nadph_c'):
    lookup = problem.simulator.metabolite_reaction_lookup(force_recalculate=True)[metabolite]
    return {r: coef * result.fluxes[r] for r, coef in lookup.items() if coef * result.fluxes[r] > 1e-6}


pd.DataFrame({'without swaps': producers(baseline), 'with swaps': producers(res)}).fillna(0).round(3) + 0.0

swapped: ['TRSARr']
constraints: {'TRSARr': (0, 0), 'TRSARr_SWAP': (-1000.0, 1000.0)}
without swaps: threonine 11.5342 mmol/gDW/h, growth 0.10 1/h, glucose uptake 10.0 mmol/gDW/h
with swaps: threonine 12.0704 mmol/gDW/h, growth 0.10 1/h, glucose uptake 10.0 mmol/gDW/h


,without swaps,with swaps
G6PDH2r,17.339,0.500
GND,17.339,0.500
ICDHyr,1.250,3.332
MTHFD,0.099,0.099
TRSARr_SWAP,0.000,33.205


Without swaps, NADPH comes mostly from the oxidative pentose phosphate pathway (G6PDH2r and GND). With the swap, the swapped copy of TRSARr supplies most of it, and the NADPH from G6PDH2r and GND falls from 17.339 to 0.500 each. The table below shows how: the reactions around TRSARr, the transhydrogenase THD2pp and the CO2 export, with their equations and fluxes.

In [7]:
reactions = ['TRSARr_SWAP', 'HPYRI', 'HPYRRx', 'THD2pp', 'G6PDH2r', 'GND', 'EX_co2_e']
pd.DataFrame({
    'reaction': {r: problem.simulator.get_reaction(r).name for r in reactions},
    'equation': {r: problem.model.reactions.get_by_id(r).reaction for r in reactions},
    'without swaps': {r: round(baseline.fluxes[r], 3) + 0.0 for r in reactions},  # + 0.0 turns -0.0 into 0.0
    'with swaps': {r: round(res.fluxes[r], 3) + 0.0 for r in reactions},
})

,reaction,equation,without swaps,with swaps
TRSARr_SWAP,Tartronate semialdehyde reductase SWAP,2h3oppan_c + h_c + nadph_c --> glyc__R_c + nadp_c,0.000,-33.205
HPYRI,Hydroxypyruvate isomerase,hpyr_c <=> 2h3oppan_c,0.000,-33.205
HPYRRx,Hydroxypyruvate reductase (NADH),h_c + hpyr_c + nadh_c --> glyc__R_c + nad_c,0.000,33.205
THD2pp,NAD(P) transhydrogenase (periplasm),2.0 h_p + nadh_c + nadp_c --> 2.0 h_c + nad_c ...,0.000,0.000
G6PDH2r,Glucose 6-phosphate dehydrogenase,g6p_c + nadp_c <=> 6pgl_c + h_c + nadph_c,17.339,0.500
GND,Phosphogluconate dehydrogenase,6pgc_c + nadp_c --> co2_c + nadph_c + ru5p__D_c,17.339,0.500
EX_co2_e,CO2 exchange,co2_e <=>,9.692,7.614


The swapped copy is written with `-->` because MEWpy adds it closed (bounds 0, 0); the swap opens it with the bounds of TRSARr (−1000, 1000), and its flux is negative: it oxidises glycerate and reduces NADP+. With hydroxypyruvate isomerase (HPYRI) and the NADH-dependent hydroxypyruvate reductase (HPYRRx) it closes a cycle whose net reaction is NADH + NADP+ → NAD+ + NADPH, a transhydrogenase that, unlike THD2pp, does not use the proton gradient (`h_p`). Little NADPH then comes from the oxidative pentose phosphate pathway, where GND releases CO2: CO2 export falls from 9.692 to 7.614 mmol/gDW/h and threonine production rises from 11.5342 to 12.0704 mmol/gDW/h at the same glucose uptake. FBA does not consider the NADH/NAD+ and NADPH/NADP+ ratios that decide whether such a cycle can carry flux in a cell.